In [1]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

ndvi_files = sorted(Path("/home/jovyan/scratch/eds-work-processing/p089r078/run-auto-scale/ga1_stage").glob("*.tif"))

dates = [
    datetime.strptime(f.stem.split("_")[2], "%Y%m%d")  # adjust to your filename
    for f in ndvi_files
]

arrays = []

for f in ndvi_files:
    with rasterio.open(f) as src:
        arr = src.read(1).astype("float32")
        arr[arr == src.nodata] = np.nan
        arrays.append(arr)

ndvi_stack = np.stack(arrays)

In [1]:
mean_ndvi = np.nanmean(ndvi_stack, axis=(1, 2))

NameError: name 'np' is not defined

In [ ]:
row, col = 500, 700
pixel_ndvi = ndvi_stack[:, row, col]

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "date": dates,
    "ndvi": mean_ndvi
})

df["doy"] = df["date"].dt.dayofyear
df["year"] = df["date"].dt.year

seasonal = df.groupby("doy")["ndvi"].agg(
    mean="mean",
    p10=lambda x: np.nanpercentile(x, 10),
    p90=lambda x: np.nanpercentile(x, 90)
).reset_index()

In [ ]:
current = df.iloc[-1]

plt.figure(figsize=(8, 4), dpi=200)

plt.fill_between(
    seasonal["doy"],
    seasonal["p10"],
    seasonal["p90"],
    alpha=0.25,
    label="Historical seasonal range"
)

plt.plot(
    seasonal["doy"],
    seasonal["mean"],
    linewidth=2,
    label="Historical seasonal mean"
)

plt.scatter(
    current["doy"],
    current["ndvi"],
    s=90,
    label="Current observation",
    zorder=5
)

plt.vlines(
    current["doy"],
    seasonal.loc[seasonal["doy"].sub(current["doy"]).abs().idxmin(), "mean"],
    current["ndvi"],
    linestyle="--",
    linewidth=2,
    label="Anomaly"
)

plt.xlabel("Day of year")
plt.ylabel("NDVI")
plt.title("NDVI seasonal behaviour and current anomaly")
plt.legend()
plt.grid(alpha=0.25)

plt.tight_layout()
plt.savefig("ndvi_seasonal_anomaly.png", transparent=True)
plt.show()

In [ ]:
selected_indices = [0, 3, 6, 9, -1]  # adjust depending on your dates

for i, idx in enumerate(selected_indices):
    plt.figure(figsize=(5, 5), dpi=200)
    plt.imshow(ndvi_stack[idx], vmin=0, vmax=1)
    plt.axis("off")
    plt.title(dates[idx].strftime("%Y-%m-%d"))
    plt.tight_layout()
    plt.savefig(f"ndvi_layer_{i+1}.png", transparent=True, bbox_inches="tight", pad_inches=0)
    plt.close()

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from pathlib import Path
from datetime import datetime

files = sorted(Path("/home/jovyan/scratch/eds-work-processing/p089r078/run-auto-scale/ga1_stage").glob("*.tif"))

dates = []
ndvi = []

for f in files:
    # adjust date parsing to your filename
    date = datetime.strptime(f.stem.split("_")[2], "%Y%m%d")
    dates.append(date)

    with rasterio.open(f) as src:
        arr = src.read(1).astype("float32")
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan

        # optional: constrain NDVI range
        arr[(arr < -1) | (arr > 1)] = np.nan

        ndvi.append(arr)

stack = np.stack(ndvi)

In [ ]:
selected = [0, 3, 6, 9, -1]

fig, ax = plt.subplots(figsize=(8, 7), facecolor="black")
ax.set_facecolor("black")
ax.axis("off")

for i, idx in enumerate(selected):
    img = stack[idx]

    x_offset = i * 0.08
    y_offset = i * 0.08

    ax.imshow(
        img,
        cmap="viridis",
        vmin=0,
        vmax=1,
        extent=[
            x_offset,
            x_offset + 1,
            y_offset,
            y_offset + 1
        ],
        alpha=0.9,
        zorder=i
    )

    ax.text(
        x_offset + 1.03,
        y_offset + 0.8,
        dates[idx].strftime("%Y-%m-%d"),
        color="white",
        fontsize=9
    )

plt.savefig(
    "ndvi_temporal_stack.png",
    dpi=300,
    transparent=True,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import pandas as pd

# whole tile mean
mean_ndvi = np.nanmean(stack, axis=(1, 2))

df = pd.DataFrame({
    "date": dates,
    "ndvi": mean_ndvi
})

df["doy"] = df["date"].dt.dayofyear

seasonal = df.groupby("doy")["ndvi"].agg(
    mean="mean",
    p10=lambda x: np.nanpercentile(x, 10),
    p90=lambda x: np.nanpercentile(x, 90)
).reset_index()

seasonal["mean_smooth"] = gaussian_filter1d(seasonal["mean"], sigma=2)
seasonal["p10_smooth"] = gaussian_filter1d(seasonal["p10"], sigma=2)
seasonal["p90_smooth"] = gaussian_filter1d(seasonal["p90"], sigma=2)